In [1]:
# Building and evaluating multiple ensemble models

In [ ]:
# Compare Ensemble models
    # -Excel in different Scenarios
    # -helps identify the most effective model for a specific problem or dataset

# Ensemble methods to consider:
    # -Bagging(eg: Random Forest)
        # -Reduces variance by averaging predictions from multiple independent models
        # -works well with high-variance models like decision trees
    # -Boosting(eg Gradient Boosting, XGBoost, LightGBM)
        # -reduces bias by sequentially training models to correct previous errors
        # -works well with low-variance models like linear models
        # -effective for complex patterns and large or imbalanced datasets
    # -Stacking
        # -combines predictions from multiple models using a meta-model
        # -can capture complex relationships and interactions
        # -requires more computational resources
        

In [3]:
# Compare Bagging and Boosting

# Bagging:
    # -builds models independently using random subsets of data
    # -Robust against overfiting with strong base learners

# Boosting:
    # -Sequentially builds models, focusing on hard-to-predict samples
    # -requires careful tuning to prevent overfitting 

In [4]:
# Model performance on Balanced vs Imbalanced data
#
# Challanges with Imbalanced Data:
    # -Models may prioritize the majority class, leading to poor performance on the minority class.
# Evaluation Metrics:
    # 1.Accuracy:
        # -May not reflect true performance for imbalanced datasets
    # 2.F1-Score:
        # -Balances precision and recall, focusing on the minority class
    # 3.ROC-AUC:
        # -Evaluates the model's ability to distinguish between classes across thresholds

In [5]:
# Ensemble learning and model comparsion
    # -Train and compare multiple ensemble models on the real-worlds dataset, analyzing their performance under balanced and imbalanced conditions

In [30]:
pip install imblearn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score

# load dataset
df = pd.read_csv("Telco-Customer-Churn.csv")

# Display dataset info and preview
print("dataset info:\n")
print(df.info())
print("\n class Distribution: \n")
print(df['churn'].value_counts())
print("\n sample data:\n")
print("\n Sample Data:\n",df.head())

# handle missing values
df['total_charges']= pd.to_numeric(df['total_charges'], errors='coerce')
df.fillna({'total_charges': df['total_charges'].median()}, inplace=True)

# Encode categorical variable
label_encoder = LabelEncoder()
for column in df.select_dtypes(include=['object']).columns:
    if column != 'churn':
        df[column] = label_encoder.fit_transform(df[column])


# Encode target variables
df['churn'] = label_encoder.fit_transform(df['churn'])

# Scale numerical features
scaler = StandardScaler()
numerical_features = ['tenure','monthly_charges','total_charges']
df[numerical_features] = scaler.fit_transform(df[numerical_features])

# Features and targets
X = df.drop(columns=['churn'])
y = df['churn']

# split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Display class distribution after SMOTE
print("\n Class Distribution after SMOTE: \n")
print(pd.Series(y_train_resampled).value_counts())

# Train Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_resampled, y_train_resampled)
y_pred_rf = rf_model.predict(X_test)
roc_auc_rf = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])

# Train XGBoost
xgb_model = XGBClassifier(eval_metric='logloss',random_state=42)
xgb_model.fit(X_train_resampled, y_train_resampled)
y_pred_xgb = xgb_model.predict(X_test)
roc_auc_xgb = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])

# Train LightGBM
lgb_model = LGBMClassifier(random_state=42)
lgb_model.fit(X_train_resampled, y_train_resampled)
y_pred_lgb = lgb_model.predict(X_test)
roc_auc_lgb = roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:, 1])

# Classification reports
print("Random Forest Report:\n", classification_report(y_test, y_pred_rf))
print("XGBoost Report: \n", classification_report(y_test, y_pred_xgb))
print("LightGBM Report: \n", classification_report(y_test, y_pred_lgb))

# ROC_AUC comparison
print("ROC-AUC Scores: \n")
print(f"Random Forest: {roc_auc_rf:.2f}")
print(f"XGBoost: {roc_auc_xgb:.2f}")
print(f"LightGBM: {roc_auc_lgb:.2f}")



dataset info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      500 non-null    int64  
 1   gender           500 non-null    object 
 2   age              500 non-null    int64  
 3   tenure           500 non-null    int64  
 4   monthly_charges  500 non-null    float64
 5   total_charges    500 non-null    float64
 6   contract_type    500 non-null    object 
 7   payment_method   500 non-null    object 
 8   churn            500 non-null    int64  
dtypes: float64(2), int64(4), object(3)
memory usage: 35.3+ KB
None

 class Distribution: 

churn
0    344
1    156
Name: count, dtype: int64

 sample data:


 Sample Data:
    customer_id  gender  age  tenure  monthly_charges  total_charges  \
0            1    Male   38       1            32.96      35.280731   
1            2  Female   49      51            87.32    482